# Option B : Classification - Prédire la DIRECTION (UP/DOWN)

**Approche classification** : Au lieu de prédire le prix exact, on prédit si le prix va monter ou descendre.

**Avantages :**
- Plus facile à apprendre (binaire vs régression)
- Plus actionnable pour le trading
- Métriques plus claires (accuracy)

**Classes :**
- UP (1): Prix de clôture demain > aujourd'hui
- DOWN (0): Prix de clôture demain < aujourd'hui

**Runtime recommandé:** GPU T4

In [ ]:
!pip install yfinance pandas numpy scikit-learn tensorflow matplotlib --quiet
print("Dependencies installed!")

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional, GRU
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras.optimizers import Adam

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ============ CONFIGURATION ============
CONFIG = {
    "symbol": "BTC-USD",
    "years_of_data": 5,
    "sequence_length": 14,           # 2 semaines - pattern court terme
    
    # Features minimales mais pertinentes
    "feature_columns": ["Open", "High", "Low", "Close", "Volume"],
    
    # Seuil pour définir UP/DOWN (éviter le bruit)
    "threshold_percent": 0.0,        # 0% = tout mouvement compte
    
    # Modèle
    "lstm_units": 128,
    "dropout_rate": 0.4,
    
    # Entraînement
    "batch_size": 64,
    "epochs": 200,
    "learning_rate": 0.0005,
    "validation_split": 0.15,
    "test_split": 0.1,
}

print("Configuration Option B (Classification):")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Télécharger les données
end_date = datetime.now()
start_date = end_date - timedelta(days=CONFIG["years_of_data"] * 365)

print(f"Downloading {CONFIG['symbol']} data...")
df = yf.download(CONFIG["symbol"], start=start_date, end=end_date, progress=False)
df = df.dropna()

print(f"Downloaded {len(df)} days")
print(f"Date range: {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}")

In [ ]:
# Créer les features additionnelles pour la classification
df_features = df.copy()

# Returns (important pour classification)
df_features['Returns'] = df_features['Close'].pct_change()
df_features['Returns_2d'] = df_features['Close'].pct_change(2)
df_features['Returns_5d'] = df_features['Close'].pct_change(5)

# Volatilité
df_features['Volatility'] = df_features['Returns'].rolling(window=7).std()

# Range normalisé
df_features['Range'] = (df_features['High'] - df_features['Low']) / df_features['Close']

# Position dans le range du jour
df_features['Close_Position'] = (df_features['Close'] - df_features['Low']) / (df_features['High'] - df_features['Low'] + 1e-8)

# Volume relatif
df_features['Volume_Ratio'] = df_features['Volume'] / df_features['Volume'].rolling(window=20).mean()

# Créer le TARGET : Direction du prix demain
df_features['Target'] = (df_features['Close'].shift(-1) > df_features['Close']).astype(int)

# Supprimer les NaN
df_features = df_features.dropna()

# Distribution des classes
print(f"\nTarget distribution:")
print(df_features['Target'].value_counts())
print(f"\nUP ratio: {df_features['Target'].mean()*100:.1f}%")

In [ ]:
# Sélectionner les features
feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 
                'Returns', 'Returns_2d', 'Returns_5d',
                'Volatility', 'Range', 'Close_Position', 'Volume_Ratio']

seq_len = CONFIG["sequence_length"]

# Extraire les données
data = df_features[feature_cols].values
targets = df_features['Target'].values

# Normalisation
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data)

# Créer les séquences
X, y = [], []
for i in range(seq_len, len(data_scaled)):
    X.append(data_scaled[i-seq_len:i])
    y.append(targets[i])

X = np.array(X)
y = np.array(y)

# Split temporel
test_size = int(len(X) * CONFIG["test_split"])
val_size = int(len(X) * CONFIG["validation_split"])
train_size = len(X) - test_size - val_size

X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:train_size+val_size], y[train_size:train_size+val_size]
X_test, y_test = X[train_size+val_size:], y[train_size+val_size:]

print(f"\nData shapes:")
print(f"  X_train: {X_train.shape}, y_train UP ratio: {y_train.mean()*100:.1f}%")
print(f"  X_val: {X_val.shape}, y_val UP ratio: {y_val.mean()*100:.1f}%")
print(f"  X_test: {X_test.shape}, y_test UP ratio: {y_test.mean()*100:.1f}%")

In [ ]:
# Class weights pour gérer le déséquilibre
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f"Class weights: {class_weight_dict}")

In [ ]:
# Modèle de Classification
model = Sequential([
    # Bidirectional LSTM
    Bidirectional(LSTM(CONFIG["lstm_units"], return_sequences=True), 
                  input_shape=(X_train.shape[1], X_train.shape[2])),
    BatchNormalization(),
    Dropout(CONFIG["dropout_rate"]),
    
    # GRU layer
    GRU(CONFIG["lstm_units"] // 2, return_sequences=False),
    BatchNormalization(),
    Dropout(CONFIG["dropout_rate"]),
    
    # Dense layers
    Dense(64, activation='relu'),
    Dropout(CONFIG["dropout_rate"] / 2),
    Dense(32, activation='relu'),
    
    # Output: Probabilité de UP
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=Adam(learning_rate=CONFIG["learning_rate"]),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=25, restore_best_weights=True, 
                  mode='max', verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_model_B.keras', monitor='val_accuracy', save_best_only=True, 
                    mode='max', verbose=0)
]

print("Training Option B (Classification UP/DOWN)...\n")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# Visualiser l'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Train')
axes[1].plot(history.history['val_accuracy'], label='Validation')
axes[1].axhline(y=0.5, color='r', linestyle='--', label='Random (50%)')
axes[1].set_title('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Évaluation
model = keras.models.load_model('best_model_B.keras')

y_pred_proba = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

print("=" * 60)
print("OPTION B - RESULTATS (Classification UP/DOWN)")
print("=" * 60)

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy*100:.2f}%")
print(f"F1-Score: {f1*100:.2f}%")
print(f"\nBaseline (random): 50%")
print(f"Improvement over random: {(accuracy - 0.5)*100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['DOWN', 'UP']))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title(f'Confusion Matrix - Accuracy: {accuracy*100:.1f}%')
plt.colorbar()

classes = ['DOWN', 'UP']
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)

# Ajouter les valeurs
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black",
                 fontsize=20)

plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Analyser la confiance des prédictions
plt.figure(figsize=(14, 5))

# Distribution des probabilités
plt.subplot(1, 2, 1)
plt.hist(y_pred_proba[y_test == 1], bins=30, alpha=0.7, label='Actual UP', color='green')
plt.hist(y_pred_proba[y_test == 0], bins=30, alpha=0.7, label='Actual DOWN', color='red')
plt.axvline(x=0.5, color='black', linestyle='--', label='Threshold')
plt.xlabel('Predicted Probability of UP')
plt.ylabel('Count')
plt.title('Distribution of Predictions')
plt.legend()

# Accuracy par niveau de confiance
plt.subplot(1, 2, 2)
thresholds = [0.5, 0.55, 0.6, 0.65, 0.7]
accuracies = []
counts = []

for thresh in thresholds:
    confident_mask = (y_pred_proba > thresh) | (y_pred_proba < 1 - thresh)
    if confident_mask.sum() > 0:
        acc = accuracy_score(y_test[confident_mask], (y_pred_proba[confident_mask] > 0.5).astype(int))
        accuracies.append(acc * 100)
        counts.append(confident_mask.sum())
    else:
        accuracies.append(0)
        counts.append(0)

x = range(len(thresholds))
plt.bar(x, accuracies, color='steelblue')
plt.xticks(x, [f'>{t}' for t in thresholds])
plt.ylabel('Accuracy %')
plt.xlabel('Confidence Threshold')
plt.title('Accuracy by Confidence Level')
plt.axhline(y=50, color='r', linestyle='--', label='Random')

# Ajouter les counts
for i, (acc, cnt) in enumerate(zip(accuracies, counts)):
    plt.text(i, acc + 1, f'n={cnt}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Sauvegarder
model.save('btc_lstm_model_B.keras')

scalers = {
    'scaler_features': scaler,
}
joblib.dump(scalers, 'btc_scaler_B.pkl')

# Métriques
metrics = {
    'accuracy': float(accuracy),
    'f1_score': float(f1),
    'improvement_over_random': float(accuracy - 0.5),
    'test_samples': int(len(y_test)),
    'up_ratio_test': float(y_test.mean()),
}

config_save = {
    'model_version': '1.0.0-B-classification',
    'trained_at': datetime.now().isoformat(),
    'option': 'B - Classification UP/DOWN',
    'sequence_length': CONFIG['sequence_length'],
    'feature_columns': feature_cols,
    'target': 'direction (UP=1, DOWN=0)',
    'metrics': metrics,
}

with open('training_config_B.json', 'w') as f:
    json.dump(config_save, f, indent=2, default=str)

print("\nFiles saved:")
print("  - btc_lstm_model_B.keras")
print("  - btc_scaler_B.pkl")
print("  - training_config_B.json")

In [ ]:
# Simulation de trading simple
print("\n" + "="*60)
print("SIMULATION DE TRADING")
print("="*60)

# Récupérer les prix réels pour la période de test
test_start_idx = train_size + val_size + seq_len
test_prices = df_features['Close'].values[test_start_idx:test_start_idx+len(y_test)]

# Stratégie: Acheter si prédit UP, vendre si prédit DOWN
initial_capital = 10000
capital = initial_capital
position = 0  # 0 = pas de position, 1 = long
trades = []

for i in range(len(y_pred) - 1):
    if y_pred[i] == 1 and position == 0:  # Signal UP, acheter
        position = 1
        entry_price = test_prices[i]
    elif y_pred[i] == 0 and position == 1:  # Signal DOWN, vendre
        exit_price = test_prices[i]
        profit_pct = (exit_price - entry_price) / entry_price
        capital *= (1 + profit_pct)
        trades.append(profit_pct)
        position = 0

# Fermer la position finale si ouverte
if position == 1:
    exit_price = test_prices[-1]
    profit_pct = (exit_price - entry_price) / entry_price
    capital *= (1 + profit_pct)
    trades.append(profit_pct)

# Buy & Hold pour comparaison
buy_hold_return = (test_prices[-1] - test_prices[0]) / test_prices[0]
buy_hold_capital = initial_capital * (1 + buy_hold_return)

print(f"\nPériode de test: {len(y_test)} jours")
print(f"Nombre de trades: {len(trades)}")
print(f"\nStratégie ML:")
print(f"  Capital final: ${capital:,.2f}")
print(f"  Return: {((capital - initial_capital) / initial_capital) * 100:.2f}%")
print(f"\nBuy & Hold:")
print(f"  Capital final: ${buy_hold_capital:,.2f}")
print(f"  Return: {buy_hold_return * 100:.2f}%")

if trades:
    winning_trades = [t for t in trades if t > 0]
    print(f"\nWin rate: {len(winning_trades)/len(trades)*100:.1f}%")
    print(f"Avg win: {np.mean(winning_trades)*100:.2f}%" if winning_trades else "")
    losing_trades = [t for t in trades if t < 0]
    print(f"Avg loss: {np.mean(losing_trades)*100:.2f}%" if losing_trades else "")

In [ ]:
# Télécharger
try:
    from google.colab import files
    files.download('btc_lstm_model_B.keras')
    files.download('btc_scaler_B.pkl')
    files.download('training_config_B.json')
except:
    print("Download manually from the file browser")